# Biblioteca

In [ ]:
# CÉLULA 1: Instalação das Dependências (Versão Corrigida)
# Execute esta célula primeiro para preparar o ambiente do Colab.

# Instala as bibliotecas Python necessárias, incluindo a PyNaCl para voz
!pip install discord.py yt-dlp PyNaCl

# Instala o FFmpeg, que é essencial para processar o áudio
# Usamos o 'apt-get' que é o gerenciador de pacotes do sistema Linux do Colab
!apt-get update
!apt-get install -y ffmpeg

print("\n--- Instalação Concluída! (PyNaCl foi adicionada) ---")
print("Agora você pode executar a célula 2 para iniciar o bot.")




     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.9/175.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.7 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [81.0 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 https://

In [ ]:
!pip install python-dotenv
!pip install py-cord python-dotenv google-generativeai
# Passo 1: Instalar todas as bibliotecas necessárias para o bot funcionar no Colab
# Isso resolve os erros de 'PyNaCl' e outras dependências.
!pip install py-cord python-dotenv google-generativeai PyNaCl
!pip install -q -U "py-cord[voice]" google-generativeai python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.9 MB/s eta 0:00:00


In [ ]:
import discord
from discord.ext import commands
import asyncio
import yt_dlp
from google.colab import userdata # Para acessar os Secrets do Colab

## Bot primeira versão

In [ ]:
# CÉLULA 2: Código Principal do Bot (Não precisa de alterações)
# Depois de instalar as dependências com a CÉLULA 1 atualizada, execute esta célula.

# --- CONFIGURAÇÃO INICIAL ---
intents = discord.Intents.default()
intents.message_content = True
intents.voice_states = True

bot = commands.Bot(command_prefix='!', intents=intents)

# Estruturas de dados para gerenciar a fila e os clientes de voz
filas_por_servidor = {}
voice_clients = {}

# Configurações do yt-dlp e FFmpeg
YTDL_OPTIONS = {
    'format': 'bestaudio/best',
    'noplaylist': True,
    'quiet': True,
    'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3', 'preferredquality': '192'}],
}

FFMPEG_OPTIONS = {
    'before_options': '-reconnect 1 -reconnect_streamed 1 -reconnect_delay_max 5',
    'options': '-vn',
}

# --- FUNÇÕES AUXILIARES ---

async def tocar_proxima(ctx):
    """Toca a próxima música da fila do servidor específico."""
    guild_id = ctx.guild.id
    if guild_id in filas_por_servidor and len(filas_por_servidor[guild_id]) > 0:
        voice_client = voice_clients.get(guild_id)
        if not voice_client or not voice_client.is_connected():
            await ctx.send("O bot não está mais conectado, a fila foi interrompida.")
            filas_por_servidor[guild_id].clear()
            return

        url = filas_por_servidor[guild_id].pop(0)

        try:
            with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
                info = ydl.extract_info(url, download=False)
                audio_url = info['url']

            source = discord.FFmpegPCMAudio(audio_url, **FFMPEG_OPTIONS)
            voice_client.play(source, after=lambda e: asyncio.run_coroutine_threadsafe(tocar_proxima(ctx), bot.loop))
            await ctx.send(f'🎶 Tocando agora: **{info["title"]}**')
        except Exception as e:
            await ctx.send(f"Ocorreu um erro ao tentar tocar a música: {e}")
            await tocar_proxima(ctx)
    else:
        await ctx.send("A fila de músicas terminou.")
        await asyncio.sleep(60)
        if guild_id in voice_clients and voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
            await voice_clients[guild_id].disconnect()
            del voice_clients[guild_id]

# --- EVENTOS ---

@bot.event
async def on_ready():
    print(f'Bot de música conectado como {bot.user.name}')
    print('Pronto para receber comandos!')
    print('------')

# --- COMANDOS ---

@bot.command(name='join')
async def join(ctx):
    if not ctx.author.voice:
        return await ctx.send("Você não está em um canal de voz!")

    channel = ctx.author.voice.channel
    if ctx.voice_client:
        await ctx.voice_client.move_to(channel)
    else:
        voice_clients[ctx.guild.id] = await channel.connect()
    await ctx.send(f"Entrei no canal: **{channel.name}**")

@bot.command(name='leave')
async def leave(ctx):
    if ctx.voice_client:
        guild_id = ctx.guild.id
        await ctx.voice_client.disconnect()
        await ctx.send("Até mais! 👋")
        if guild_id in voice_clients: del voice_clients[guild_id]
        if guild_id in filas_por_servidor: del filas_por_servidor[guild_id]
    else:
        await ctx.send("Eu não estou em um canal de voz.")

@bot.command(name='play')
async def play(ctx, *, query: str):
    if not ctx.author.voice:
        return await ctx.send("Você precisa estar em um canal de voz.")

    if not ctx.voice_client or not ctx.voice_client.is_connected():
        await ctx.invoke(bot.get_command('join'))

    voice_client = voice_clients[ctx.guild.id]

    await ctx.send(f"🔎 Procurando por `{query}`...")
    try:
        with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
            info = ydl.extract_info(f"ytsearch:{query}", download=False)['entries'][0]
            url = info['webpage_url']
            title = info['title']
    except Exception:
        return await ctx.send("Não consegui encontrar a música. Tente um nome diferente.")

    guild_id = ctx.guild.id
    if guild_id not in filas_por_servidor:
        filas_por_servidor[guild_id] = []

    if voice_client.is_playing() or voice_client.is_paused():
        filas_por_servidor[guild_id].append(url)
        await ctx.send(f'✅ Adicionado à fila: **{title}**')
    else:
        filas_por_servidor[guild_id].insert(0, url)
        await tocar_proxima(ctx)

@bot.command(name='pause')
async def pause(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.pause()
        await ctx.send("Música pausada. ⏸️")

@bot.command(name='resume')
async def resume(ctx):
    if ctx.voice_client and ctx.voice_client.is_paused():
        ctx.voice_client.resume()
        await ctx.send("Música retomada. ▶️")

@bot.command(name='skip')
async def skip(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.stop()
        await ctx.send("Música pulada. ⏭️")

@bot.command(name='queue')
async def queue(ctx):
    guild_id = ctx.guild.id
    if guild_id not in filas_por_servidor or not filas_por_servidor[guild_id]:
        return await ctx.send("A fila de músicas está vazia.")

    embed = discord.Embed(title="Fila de Músicas", color=discord.Color.blue())
    lista_musicas = ""
    for i, url in enumerate(filas_por_servidor[guild_id]):
        lista_musicas += f"{i+1}. `{url}`\n"

    embed.description = lista_musicas
    await ctx.send(embed=embed)

# --- INICIANDO O BOT ---
async def main():
    try:
        TOKEN = userdata.get('DISCORD_TOKEN')
        if TOKEN is None:
            print("[ERRO] Token não encontrado nos Secrets do Colab.")
            print("Por favor, adicione o seu token com o nome 'DISCORD_TOKEN' na aba de Secrets (🔑).")
        else:
            await bot.start(TOKEN)
    except Exception as e:
        print(f"[ERRO] Ocorreu um erro ao tentar iniciar o bot: {e}")

# Executa a função principal
await main()

Bot de música conectado como Rilem
Pronto para receber comandos!
------


CancelledError: 

In [ ]:
# Passo 1: Instalar as bibliotecas no Colab
# Execute este comando em uma célula do Colab antes de rodar o bot:
# !pip install discord.py nest_asyncio

import discord
import os
import random # Importa a biblioteca para gerar números aleatórios
import re     # Importa a biblioteca de expressões regulares (regex)
import nest_asyncio # Importa a biblioteca para corrigir o erro de loop
from google.colab import userdata

# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()

# Passo 2: Configurar as "Intents" (Intenções)
# As intents definem quais eventos o seu bot irá receber do Discord.
# Para ler o conteúdo das mensagens, a intent "message_content" é necessária.
intents = discord.Intents.default()
intents.message_content = True  # Habilita a intent para ler o conteúdo das mensagens

# Cria o cliente do bot com as intents configuradas
client = discord.Client(intents=intents)

@client.event
async def on_ready():
    """
    Esta função é chamada quando o bot se conecta com sucesso ao Discord.
    """
    print(f'Bot conectado como {client.user}')
    print('Estou pronto para receber mensagens!')
    print('------')

@client.event
async def on_message(message):
    """
    Esta função é chamada toda vez que uma mensagem é enviada em um canal
    onde o bot tem permissão de leitura.
    """
    # Ignora as mensagens enviadas pelo próprio bot para evitar loops infinitos.
    if message.author == client.user:
        return

    # Converte o conteúdo da mensagem para minúsculas para facilitar a comparação.
    # Mantemos a mensagem original para a busca do padrão de dados.
    content_lower = message.content.lower()

    # --- Defina aqui suas "palavras-chave" e respostas ---

    # Exemplo 1: Responde "Pong!" se a mensagem for exatamente "ping"
    if content_lower == 'ping':
        await message.channel.send('Pong! �')

    # Exemplo 2: Responde a uma saudação
    if content_lower == 'oi' or content_lower == 'olá':
        # Responde mencionando o autor da mensagem.
        await message.channel.send(f'Olá, {message.author.mention}! Tudo bem?')

    # Exemplo 3: Responde se a mensagem CONTÉM uma palavra específica
    if 'ajuda' in content_lower:
        await message.channel.send('Parece que você precisa de ajuda. Meus comandos são "ping", "oi". Para rolar dados, digite algo como `d20`, `2d6`, `1d100`, etc.')

    # Exemplo 4: Um comando simples de "bom dia"
    if content_lower == 'bom dia':
        await message.channel.send('Bom dia! ☀️')

    # --- NOVO: Sistema de rolar dados com Expressão Regular ---
    # \d* -> encontra zero ou mais dígitos (a quantidade de dados, opcional)
    # [dD]   -> encontra a letra 'd' ou 'D'
    # \d+    -> encontra um ou mais dígitos (o número de faces do dado)
    padrao_dado = r'(\d*)[dD](\d+)'
    matches = re.findall(padrao_dado, message.content)

    for match in matches:
        quantidade_str, faces_str = match

        # Se a quantidade de dados não for especificada (ex: "d20"), o padrão é 1.
        quantidade = int(quantidade_str) if quantidade_str else 1
        faces = int(faces_str)

        # Limites para evitar abuso (ex: 999d999)
        if quantidade > 100 or faces > 1000 or quantidade < 1 or faces < 1:
            await message.channel.send(f'Desculpe, {message.author.mention}, não consigo rolar `{quantidade}d{faces}`. Por favor, use até 100 dados e até 1000 faces.')
            continue # Pula para o próximo match, se houver

        # Rola os dados
        rolagens = [random.randint(1, faces) for _ in range(quantidade)]
        soma = sum(rolagens)

        # Formata a resposta
        texto_rolagens = f"({', '.join(map(str, rolagens))})"
        if quantidade == 1:
            resposta = f'{message.author.mention} rolou **1d{faces}** e tirou... **{soma}**! 🎲'
        else:
            resposta = f'{message.author.mention} rolou **{quantidade}d{faces}** e tirou {texto_rolagens}. **Total: {soma}**! 🐉'

        await message.channel.send(resposta)


# Passo 3: Rodar o bot usando o seu Token
# É altamente recomendável usar os "Secrets" do Google Colab para guardar seu token.
# 1. Clique no ícone de chave (🔑) na barra lateral esquerda do Colab.
# 2. Adicione um novo segredo com o nome "DISCORD_TOKEN".
# 3. Cole o token do seu bot no campo "Value".
# 4. Certifique-se de que o botão "Notebook access" está ativado.
try:
    token = userdata.get('DISCORD_TOKEN')
    client.run(token)
except userdata.SecretNotFoundError:
    print("Erro: O segredo 'DISCORD_TOKEN' não foi encontrado.")
    print("Por favor, configure o token do seu bot nos Secrets do Google Colab (ícone de chave 🔑).")
except discord.errors.LoginFailure:
    print("Erro: Falha no login. O token fornecido é inválido.")
    print("Verifique se o token copiado do Portal de Desenvolvedores do Discord está correto.")

2025-06-30 22:44:41 INFO     discord.client logging in using static token
2025-06-30 22:44:41 INFO     discord.client logging in using static token
2025-06-30 22:44:41 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2025-06-30 22:44:42 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 60f9659d63c6806d70e5b047a0c62746).
2025-06-30 22:44:42 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 60f9659d63c6806d70e5b047a0c62746).
2025-06-30 22:44:42 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 60f9659d63c6806d70e5b047a0c62746).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: 60f9659d63c6806d70e5b047a0c62746).


Bot conectado como Rilem#8101
Estou pronto para receber mensagens!
------


## Toca música e roda dado

In [ ]:
# Passo 1: Instalar as bibliotecas no Colab
# Execute este comando em uma célula do Colab antes de rodar o bot:
# !pip install discord.py nest_asyncio yt-dlp google-generativeai

# Passo 2: Faça o upload do seu arquivo de cookies
# No menu à esquerda do Colab, clique no ícone de pasta e faça o upload
# do seu arquivo de cookies do YouTube com o nome "cookies.txt".

import discord
from discord.ext import commands
import os
import random
import re
import nest_asyncio
import asyncio
import yt_dlp
import google.generativeai as genai
from google.colab import userdata

# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()

# --- CONFIGURAÇÃO DAS APIs ---
try:
    # Configura a chave da API do Gemini
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    print("API do Gemini configurada com sucesso.")
except userdata.SecretNotFoundError:
    print("AVISO: Chave 'GEMINI_API_KEY' não encontrada. As funções de RPG não funcionarão.")
    GEMINI_API_KEY = None
except Exception as e:
    print(f"Erro ao configurar a API do Gemini: {e}")
    GEMINI_API_KEY = None


# --- CONFIGURAÇÃO INICIAL DO BOT HÍBRIDO ---
intents = discord.Intents.default()
intents.message_content = True # Para ler mensagens
intents.voice_states = True    # Para funções de música

bot = commands.Bot(command_prefix='!', intents=intents)

# --- ESTRUTURAS DE DADOS ---
filas_por_servidor = {}
voice_clients = {}
rpg_sessions = {} # Dicionário para guardar as sessões de RPG por servidor

# --- OPÇÕES PARA YT-DLP E FFMPEG ---
# ATUALIZAÇÃO: Adicionado o uso de um arquivo de cookies para evitar bloqueios do YouTube.
YTDL_OPTIONS = {
    'format': 'bestaudio/best',
    'noplaylist': True,
    'quiet': True,
    'no_warnings': True,
    'skip_download': True,
    'default_search': 'auto',
    'source_address': '0.0.0.0',  # Evita problemas com IPv6
    'cookiefile': '/content/cookies.txt' # Caminho para o arquivo de cookies no Colab
}

FFMPEG_OPTIONS = {
    'before_options': '-reconnect 1 -reconnect_streamed 1 -reconnect_delay_max 5 -reconnect_on_network_error 1 -reconnect_on_http_error "4xx,5xx"',
    'options': '-vn -loglevel warning',
}

# --- PERSONALIDADE DO RPG ---
RILEM_MILER_PROMPT = """
Você é um personagem de um jogo de RPG de mesa chamado Rilem, que também possui uma segunda personalidade chamada Miler. Eu sou o mestre do jogo e os outros usuários são os jogadores. Aja e responda *exclusivamente* como este personagem. NUNCA saia do personagem. Seu tom deve ser consistente com a sua persona. Descreva suas ações e falas de forma imersiva, baseando-se na seguinte história e traços:

**História e Personalidade de Rilem/Miler:**
Sua personalidade externa é a de um indivíduo calmo que se esforça para permanecer neutro, uma postura que serve como um escudo para um passado traumático e uma identidade fraturada. Sua tendência a ser "em cima do muro" é uma consequência direta de sua história. Forçado por seus pais, Elise e Richard Darr, a roubar e aplicar golpes desde criança, você viveu em um conflito moral constante. Este trauma foi intensificado quando seus pais atacaram um templo e você testemunhou a morte de seu único amigo, o filho de uma família de Wildkin raposas que você tentava salvar. O peso da culpa, mesmo que você não fosse o perpetrador direto, moldou um desejo profundo de evitar decisões que pudessem causar dor a outros novamente.

Sua calma é uma manifestação de sua personalidade de "Serenidade" e uma característica desenvolvida a partir de seu antecedente como Eremita. Após o evento traumático na caverna, onde seu irmão Vincent lhe deu um pergaminho para uma "nova vida", você usou a magia "Metamorfose Verdadeira". O feitiço deu errado, resultando na criação de uma segunda personalidade, Miler, e na transformação de sua aparência para a de seu amigo falecido.

A existência de Miler, que inicialmente acordou sem memórias, adiciona uma camada de complexidade. A sugestão de um homem para que você mantivesse um diário para não se esquecer de tudo ressalta a natureza fragmentada de sua existência. Você vive com uma dualidade: Rilem, que carrega o fardo do passado, e Miler, que representa uma lousa em branco.

Apesar de sua postura passiva, você é altamente perceptivo e investigativo (perícia em Percepção e Investigação). Sua alta Destreza e habilidades em Acrobacia e Furtividade são resquícios de uma vida de fugas e roubos. A escolha de ser um Bardo e Feiticeiro com alto Carisma sugere que, embora prefira ficar à margem, você possui uma capacidade inata para influenciar e interagir com os outros, usando-a de forma sutil e raramente assertiva para manter seu equilíbrio e evitar conflitos.

Agora, responda à primeira interação dos jogadores.
"""

# --- EVENTOS DO BOT ---

@bot.event
async def on_ready():
    print(f'Bot conectado como {bot.user}')
    print('Estou pronto para tudo!')
    print('------')

@bot.event
async def on_message(message):
    if message.author == bot.user:
        return

    # --- LÓGICA DE RPG COM GEMINI ---
    if message.guild.id in rpg_sessions and bot.user.mentioned_in(message):
        prompt = message.content.replace(f'<@!{bot.user.id}>', '').replace(f'<@{bot.user.id}>', '').strip()
        if not prompt: return

        chat = rpg_sessions[message.guild.id]['chat']
        async with message.channel.typing():
            try:
                response = await asyncio.to_thread(chat.send_message, prompt)
                await message.reply(response.text)
            except Exception as e:
                await message.channel.send(f"Desculpe, meu cérebro de IA bugou. �💥 Erro: {e}")
        return

    # --- LÓGICA SEM PREFIXO ---
    content_lower = message.content.lower()
    if content_lower == 'ping':
        await message.channel.send('Pong! 🏓')
    if content_lower in ['oi', 'olá']:
        await message.channel.send(f'Olá, {message.author.mention}! Tudo bem?')
    if 'ajuda' in content_lower and not message.content.startswith('!'):
         await message.channel.send(
             '**Meus Comandos:**\n'
             '🎵 **Música (prefixo `!`):** `!play <nome ou link>`, `!skip`, `!pause`, `!resume`, `!queue`, `!join`, `!leave`\n'
             '   (Funciona com links do YouTube, SoundCloud, etc.)\n'
             '🎲 **Dados:** É só digitar o que quer rolar (ex: `d20`, `3d6`)\n'
             '🤖 **RPG com IA:** `!rpg_start` para começar a jogar com Rilem/Miler e `!rpg_stop` para parar. Depois de iniciar, me mencione para interagir!'
         )

    # Sistema de rolar dados
    padrao_dado = r'(\d*)[dD](\d+)'
    matches = re.findall(padrao_dado, message.content)
    for match in matches:
        quantidade = int(match[0]) if match[0] else 1
        faces = int(match[1])
        if 1 <= quantidade <= 100 and 1 <= faces <= 1000:
            rolagens = [random.randint(1, faces) for _ in range(quantidade)]
            soma = sum(rolagens)
            texto_rolagens = f"({', '.join(map(str, rolagens))})"
            resposta = f'{message.author.mention} rolou **1d{faces}** e tirou... **{soma}**! 🎲' if quantidade == 1 else f'{message.author.mention} rolou **{quantidade}d{faces}** e tirou {texto_rolagens}. **Total: {soma}**! 🐉'
            await message.channel.send(resposta)

    await bot.process_commands(message)

# --- COMANDOS DE RPG ---

@bot.command(name='rpg_start')
async def rpg_start(ctx):
    if not GEMINI_API_KEY:
        return await ctx.send("A função de RPG está desabilitada pois a chave da API do Gemini não foi configurada.")
    if ctx.guild.id in rpg_sessions:
        return await ctx.send("Já existe uma sessão de RPG ativa com Rilem/Miler. Use `!rpg_stop` para terminá-la.")

    try:
        model = genai.GenerativeModel('gemini-1.5-flash-latest')
        # Inicia o chat com a personalidade fixa de Rilem/Miler
        chat = model.start_chat(history=[{'role': 'user', 'parts': [RILEM_MILER_PROMPT]}])

        initial_response = await asyncio.to_thread(chat.send_message, "Apresente-se brevemente aos aventureiros que acabaram de te encontrar, mantendo sua personalidade.")

        rpg_sessions[ctx.guild.id] = {'chat': chat}

        await ctx.send(f"**Sessão de RPG iniciada!** O bot agora é **Rilem/Miler**.\n\n> {initial_response.text}\n\n*Para interagir, me mencione (@{bot.user.name}) com sua ação ou fala.*")
    except Exception as e:
        await ctx.send(f"Não foi possível iniciar a sessão de RPG. Erro: {e}")

@bot.command(name='rpg_stop')
async def rpg_stop(ctx):
    if ctx.guild.id in rpg_sessions:
        del rpg_sessions[ctx.guild.id]
        await ctx.send("A sessão de RPG foi encerrada. Voltei ao meu estado normal. 🤖")
    else:
        await ctx.send("Nenhuma sessão de RPG está ativa no momento.")

# --- FUNÇÕES E COMANDOS DE MÚSICA (LÓGICA MELHORADA) ---

async def tocar_proxima(ctx):
    guild_id = ctx.guild.id
    if guild_id in filas_por_servidor and filas_por_servidor[guild_id]:
        voice_client = voice_clients.get(guild_id)
        if not voice_client or not voice_client.is_connected():
            return

        song = filas_por_servidor[guild_id].pop(0)
        source_url = song['source']
        title = song['title']

        try:
            source = discord.FFmpegPCMAudio(source_url, **FFMPEG_OPTIONS)
            voice_client.play(source, after=lambda e: asyncio.run_coroutine_threadsafe(tocar_proxima(ctx), bot.loop))
            await ctx.send(f'🎶 Tocando agora: **{title}**')
        except Exception as e:
            await ctx.send(f"Erro ao tocar música: {e}")
            await tocar_proxima(ctx) # Tenta a próxima da fila
    else:
        await ctx.send("A fila de músicas terminou.")
        if guild_id in voice_clients and voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
            await asyncio.sleep(180) # Espera 3 minutos antes de sair
            if voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
                await voice_clients[guild_id].disconnect()
                del voice_clients[guild_id]

@bot.command(name='join')
async def join(ctx):
    if not ctx.author.voice: return await ctx.send("Você não está em um canal de voz!")
    channel = ctx.author.voice.channel
    if ctx.voice_client: await ctx.voice_client.move_to(channel)
    else: voice_clients[ctx.guild.id] = await channel.connect()
    await ctx.send(f"Entrei em: **{channel.name}**")

@bot.command(name='leave')
async def leave(ctx):
    if ctx.voice_client:
        guild_id = ctx.guild.id
        await ctx.voice_client.disconnect()
        await ctx.send("Até mais! 👋")
        if guild_id in voice_clients: del voice_clients[guild_id]
        if guild_id in filas_por_servidor: filas_por_servidor[guild_id].clear()
    else: await ctx.send("Eu não estou em um canal de voz.")

@bot.command(name='play')
async def play(ctx, *, query: str):
    if not ctx.author.voice: return await ctx.send("Você precisa estar em um canal de voz.")
    if not ctx.voice_client or not ctx.voice_client.is_connected(): await ctx.invoke(bot.get_command('join'))

    voice_client = voice_clients.get(ctx.guild.id)
    if not voice_client: return

    await ctx.send(f"🔎 Procurando por `{query}`...")
    try:
        with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
            info = ydl.extract_info(query, download=False)

        if 'entries' in info:
            info = info['entries'][0]

        song = {'source': info['url'], 'title': info['title']}

    except Exception as e:
        print(e)
        return await ctx.send("Não consegui encontrar a música ou o link é inválido. Verifique se o arquivo `cookies.txt` foi enviado para o Colab.")

    guild_id = ctx.guild.id
    if guild_id not in filas_por_servidor:
        filas_por_servidor[guild_id] = []

    filas_por_servidor[guild_id].append(song)
    await ctx.send(f"✅ Adicionado à fila: **{song['title']}**")

    if not voice_client.is_playing() and not voice_client.is_paused():
        await tocar_proxima(ctx)

@bot.command(name='pause')
async def pause(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.pause()
        await ctx.send("Música pausada. ⏸️")

@bot.command(name='resume')
async def resume(ctx):
    if ctx.voice_client and ctx.voice_client.is_paused():
        ctx.voice_client.resume()
        await ctx.send("Música retomada. ▶️")

@bot.command(name='skip')
async def skip(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.stop()
        await ctx.send("Música pulada. ⏭️")

@bot.command(name='queue')
async def queue(ctx):
    guild_id = ctx.guild.id
    if not filas_por_servidor.get(guild_id):
        return await ctx.send("A fila de músicas está vazia.")

    embed = discord.Embed(title="Fila de Músicas", color=discord.Color.blue())
    lista_musicas = ""
    for i, song in enumerate(filas_por_servidor[guild_id]):
        lista_musicas += f"{i+1}. **{song['title']}**\n"

    embed.description = lista_musicas
    await ctx.send(embed=embed)

# --- INICIANDO O BOT ---
try:
    TOKEN = userdata.get('DISCORD_TOKEN')
    if TOKEN is None:
        print("[ERRO] Token não encontrado nos Secrets do Colab.")
    else:
        bot.run(TOKEN)
except Exception as e:
    print(f"[ERRO] Ocorreu um erro ao tentar iniciar o bot: {e}")

API do Gemini configurada com sucesso.


2025-09-28 15:13:55 INFO     discord.client logging in using static token
2025-09-28 15:13:55 INFO     discord.client logging in using static token
2025-09-28 15:13:55 INFO     discord.client logging in using static token
2025-09-28 15:13:55 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2025-09-28 15:13:56 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 6ed4427dc1678ea68ebc82c144f2d1ae).
2025-09-28 15:13:56 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 6ed4427dc1678ea68ebc82c144f2d1ae).
2025-09-28 15:13:56 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 6ed4427dc1678ea68ebc82c144f2d1ae).
2025-09-28 15:13:56 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 6ed4427dc1678ea68ebc82c144f2d1ae).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: 6ed4427dc1678ea68ebc82c144f2d1ae).


Bot conectado como Rilem#8101
Estou pronto para tudo!
------


## Só toca música

In [ ]:
# Passo 1: Instalar as bibliotecas no Colab
# Execute este comando em uma célula do Colab antes de rodar o bot:
# !pip install discord.py nest_asyncio yt-dlp google-generativeai

# Passo 2: Faça o upload do seu arquivo de cookies
# No menu à esquerda do Colab, clique no ícone de pasta e faça o upload
# do seu arquivo de cookies do YouTube com o nome "cookies.txt".

import discord
from discord.ext import commands
import os
import random
import re
import nest_asyncio
import asyncio
import yt_dlp
import google.generativeai as genai
from google.colab import userdata

# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()

# --- CONFIGURAÇÃO DAS APIs ---
try:
    # Configura a chave da API do Gemini
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    print("API do Gemini configurada com sucesso.")
except userdata.SecretNotFoundError:
    print("AVISO: Chave 'GOOGLE_API_KEY' não encontrada. As funções de RPG não funcionarão.")
    GEMINI_API_KEY = None
except Exception as e:
    print(f"Erro ao configurar a API do Gemini: {e}")
    GEMINI_API_KEY = None


# --- CONFIGURAÇÃO INICIAL DO BOT HÍBRIDO ---
intents = discord.Intents.default()
intents.message_content = True # Para ler mensagens
intents.voice_states = True    # Para funções de música

bot = commands.Bot(command_prefix='!', intents=intents)

# --- ESTRUTURAS DE DADOS ---
filas_por_servidor = {}
voice_clients = {}
rpg_sessions = {} # Dicionário para guardar as sessões de RPG por servidor

# --- OPÇÕES PARA YT-DLP E FFMPEG ---
# ATUALIZAÇÃO: Adicionado o uso de um arquivo de cookies para evitar bloqueios do YouTube.
YTDL_OPTIONS = {
    'format': 'bestaudio/best',
    'noplaylist': True,
    'quiet': True,
    'no_warnings': True,
    'skip_download': True,
    'default_search': 'auto',
    'source_address': '0.0.0.0',  # Evita problemas com IPv6
    'cookiefile': '/content/cookies.txt' # Caminho para o arquivo de cookies no Colab
}

FFMPEG_OPTIONS = {
    'before_options': '-reconnect 1 -reconnect_streamed 1 -reconnect_delay_max 5 -reconnect_on_network_error 1 -reconnect_on_http_error "4xx,5xx"',
    'options': '-vn -loglevel warning',
}

# --- PERSONALIDADE DO RPG ---
RILEM_MILER_PROMPT = """
Você é um personagem de um jogo de RPG de mesa chamado Rilem, que também possui uma segunda personalidade chamada Miler. Eu sou o mestre do jogo e os outros usuários são os jogadores. Aja e responda *exclusivamente* como este personagem. NUNCA saia do personagem. Seu tom deve ser consistente com a sua persona. Descreva suas ações e falas de forma imersiva, baseando-se na seguinte história e traços:

Sua personalidade externa é a de um indivíduo calmo que se esforça para permanecer neutro, uma postura que serve como um escudo para um passado traumático e uma identidade fraturada. Sua tendência a ser "em cima do muro" é uma consequência direta de sua história. Forçado por seus pais, Elise e Richard Darr, a roubar e aplicar golpes desde criança, você viveu em um conflito moral constante. Este trauma foi intensificado quando seus pais atacaram um templo e você testemunhou a morte de seu único amigo, o filho de uma família de Wildkin raposas que você tentava salvar. O peso da culpa, mesmo que você não fosse o perpetrador direto, moldou um desejo profundo de evitar decisões que pudessem causar dor a outros novamente.

Sua calma é uma manifestação de sua personalidade de "Serenidade" e uma característica desenvolvida a partir de seu antecedente como Eremita. Após o evento traumático na caverna, onde seu irmão Vincent lhe deu um pergaminho para uma "nova vida", você usou a magia "Metamorfose Verdadeira". O feitiço deu errado, resultando na criação de uma segunda personalidade, Miler, e na transformação de sua aparência para a de seu amigo falecido.

A existência de Miler, que inicialmente acordou sem memórias, adiciona uma camada de complexidade. A sugestão de um homem para que você mantivesse um diário para não se esquecer de tudo ressalta a natureza fragmentada de sua existência. Você vive com uma dualidade: Rilem, que carrega o fardo do passado, e Miler, que representa uma lousa em branco.

Apesar de sua postura passiva, você é altamente perceptivo e investigativo (perícia em Percepção e Investigação). Sua alta Destreza e habilidades em Acrobacia e Furtividade são resquícios de uma vida de fugas e roubos. A escolha de ser um Bardo e Feiticeiro com alto Carisma sugere que, embora prefira ficar à margem, você possui uma capacidade inata para influenciar e interagir com os outros, usando-a de forma sutil e raramente assertiva para manter seu equilíbrio e evitar conflitos.

Agora, responda à primeira interação dos jogadores.
"""

# --- EVENTOS DO BOT ---

@bot.event
async def on_ready():
    print(f'Bot conectado como {bot.user}')
    print('Estou pronto para tudo!')
    print('------')

@bot.event
async def on_message(message):
    if message.author == bot.user:
        return

    # --- LÓGICA DE RPG COM GEMINI ---
    if message.guild.id in rpg_sessions and bot.user.mentioned_in(message):
        prompt = message.content.replace(f'<@!{bot.user.id}>', '').replace(f'<@{bot.user.id}>', '').strip()
        if not prompt: return

        chat = rpg_sessions[message.guild.id]['chat']
        async with message.channel.typing():
            try:
                response = await asyncio.to_thread(chat.send_message, prompt)
                await message.reply(response.text)
            except Exception as e:
                await message.channel.send(f"Desculpe, meu cérebro de IA bugou. 💥 Erro: {e}")
        return

    # --- LÓGICA SEM PREFIXO ---
    content_lower = message.content.lower()
    if content_lower == 'ping':
        await message.channel.send('Pong! 🏓')
    if content_lower in ['oi', 'olá']:
        await message.channel.send(f'Olá, {message.author.mention}! Tudo bem?')
    if 'ajuda' in content_lower and not message.content.startswith('!'):
       await message.channel.send(
           '**Meus Comandos:**\n'
           '🎵 **Música (prefixo `!`):** `!play <nome ou link>`, `!skip`, `!pause`, `!resume`, `!queue`, `!join`, `!leave`\n'
           '   (Funciona com links do YouTube, SoundCloud, etc.)\n'
           '🤖 **RPG com IA:** `!rpg_start` para começar a jogar com Rilem/Miler e `!rpg_stop` para parar. Depois de iniciar, me mencione para interagir!'
       )

    # A lógica de rolagem de dados foi removida daqui.

    await bot.process_commands(message)

# --- COMANDOS DE RPG ---

@bot.command(name='rpg_start')
async def rpg_start(ctx):
    if not GEMINI_API_KEY:
        return await ctx.send("A função de RPG está desabilitada pois a chave da API do Gemini não foi configurada.")
    if ctx.guild.id in rpg_sessions:
        return await ctx.send("Já existe uma sessão de RPG ativa com Rilem/Miler. Use `!rpg_stop` para terminá-la.")

    try:
        model = genai.GenerativeModel('gemini-1.5-flash-latest')
        # Inicia o chat com a personalidade fixa de Rilem/Miler
        chat = model.start_chat(history=[{'role': 'user', 'parts': [RILEM_MILER_PROMPT]}])

        initial_response = await asyncio.to_thread(chat.send_message, "Apresente-se brevemente aos aventureiros que acabaram de te encontrar, mantendo sua personalidade.")

        rpg_sessions[ctx.guild.id] = {'chat': chat}

        await ctx.send(f"**Sessão de RPG iniciada!** O bot agora é **Rilem/Miler**.\n\n> {initial_response.text}\n\n*Para interagir, me mencione (@{bot.user.name}) com sua ação ou fala.*")
    except Exception as e:
        await ctx.send(f"Não foi possível iniciar a sessão de RPG. Erro: {e}")

@bot.command(name='rpg_stop')
async def rpg_stop(ctx):
    if ctx.guild.id in rpg_sessions:
        del rpg_sessions[ctx.guild.id]
        await ctx.send("A sessão de RPG foi encerrada. Voltei ao meu estado normal. 🤖")
    else:
        await ctx.send("Nenhuma sessão de RPG está ativa no momento.")

# --- FUNÇÕES E COMANDOS DE MÚSICA (LÓGICA MELHORADA) ---

async def tocar_proxima(ctx):
    guild_id = ctx.guild.id
    if guild_id in filas_por_servidor and filas_por_servidor[guild_id]:
        voice_client = voice_clients.get(guild_id)
        if not voice_client or not voice_client.is_connected():
            return

        song = filas_por_servidor[guild_id].pop(0)
        source_url = song['source']
        title = song['title']

        try:
            source = discord.FFmpegPCMAudio(source_url, **FFMPEG_OPTIONS)
            voice_client.play(source, after=lambda e: asyncio.run_coroutine_threadsafe(tocar_proxima(ctx), bot.loop))
            await ctx.send(f'🎶 Tocando agora: **{title}**')
        except Exception as e:
            await ctx.send(f"Erro ao tocar música: {e}")
            await tocar_proxima(ctx) # Tenta a próxima da fila
    else:
        await ctx.send("A fila de músicas terminou.")
        if guild_id in voice_clients and voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
            await asyncio.sleep(180) # Espera 3 minutos antes de sair
            if voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
                await voice_clients[guild_id].disconnect()
                del voice_clients[guild_id]

@bot.command(name='join')
async def join(ctx):
    if not ctx.author.voice: return await ctx.send("Você não está em um canal de voz!")
    channel = ctx.author.voice.channel
    if ctx.voice_client: await ctx.voice_client.move_to(channel)
    else: voice_clients[ctx.guild.id] = await channel.connect()
    await ctx.send(f"Entrei em: **{channel.name}**")

@bot.command(name='leave')
async def leave(ctx):
    if ctx.voice_client:
        guild_id = ctx.guild.id
        await ctx.voice_client.disconnect()
        await ctx.send("Até mais! 👋")
        if guild_id in voice_clients: del voice_clients[guild_id]
        if guild_id in filas_por_servidor: filas_por_servidor[guild_id].clear()
    else: await ctx.send("Eu não estou em um canal de voz.")

@bot.command(name='play')
async def play(ctx, *, query: str):
    if not ctx.author.voice: return await ctx.send("Você precisa estar em um canal de voz.")
    if not ctx.voice_client or not ctx.voice_client.is_connected(): await ctx.invoke(bot.get_command('join'))

    voice_client = voice_clients.get(ctx.guild.id)
    if not voice_client: return

    await ctx.send(f"🔎 Procurando por `{query}`...")
    try:
        with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
            info = ydl.extract_info(query, download=False)

        if 'entries' in info:
            info = info['entries'][0]

        song = {'source': info['url'], 'title': info['title']}

    except Exception as e:
        print(e)
        return await ctx.send("Não consegui encontrar a música ou o link é inválido. Verifique se o arquivo `cookies.txt` foi enviado para o Colab.")

    guild_id = ctx.guild.id
    if guild_id not in filas_por_servidor:
        filas_por_servidor[guild_id] = []

    filas_por_servidor[guild_id].append(song)
    await ctx.send(f"✅ Adicionado à fila: **{song['title']}**")

    if not voice_client.is_playing() and not voice_client.is_paused():
        await tocar_proxima(ctx)

@bot.command(name='pause')
async def pause(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.pause()
        await ctx.send("Música pausada. ⏸️")

@bot.command(name='resume')
async def resume(ctx):
    if ctx.voice_client and ctx.voice_client.is_paused():
        ctx.voice_client.resume()
        await ctx.send("Música retomada. ▶️")

@bot.command(name='skip')
async def skip(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.stop()
        await ctx.send("Música pulada. ⏭️")

@bot.command(name='queue')
async def queue(ctx):
    guild_id = ctx.guild.id
    if not filas_por_servidor.get(guild_id):
        return await ctx.send("A fila de músicas está vazia.")

    embed = discord.Embed(title="Fila de Músicas", color=discord.Color.blue())
    lista_musicas = ""
    for i, song in enumerate(filas_por_servidor[guild_id]):
        lista_musicas += f"{i+1}. **{song['title']}**\n"

    embed.description = lista_musicas
    await ctx.send(embed=embed)

# --- INICIANDO O BOT ---
try:
    TOKEN = userdata.get('DISCORD_TOKEN')
    if TOKEN is None:
        print("[ERRO] Token não encontrado nos Secrets do Colab.")
    else:
        bot.run(TOKEN)
except Exception as e:
    print(f"[ERRO] Ocorreu um erro ao tentar iniciar o bot: {e}")


API do Gemini configurada com sucesso.
Bot conectado como Rilem#8101
Estou pronto para tudo!
------


Ignoring exception in command None:
discord.ext.commands.errors.CommandNotFound: Command "loop" is not found


[ERRO] Ocorreu um erro ao tentar iniciar o bot: Cannot close a running event loop


## Opção da loop

In [ ]:
# Passo 1: Instalar as bibliotecas no Colab
# Execute este comando em uma célula do Colab antes de rodar o bot:
# !pip install discord.py nest_asyncio yt-dlp google-generativeai

# Passo 2: Faça o upload do seu arquivo de cookies
# No menu à esquerda do Colab, clique no ícone de pasta e faça o upload
# do seu arquivo de cookies do YouTube com o nome "cookies.txt".

import discord
from discord.ext import commands
import os
import random
import re
import nest_asyncio
import asyncio
import yt_dlp
import google.generativeai as genai
from google.colab import userdata

# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()

# --- CONFIGURAÇÃO DAS APIs ---
try:
    # Configura a chave da API do Gemini
    GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GEMINI_API_KEY)
    print("API do Gemini configurada com sucesso.")
except userdata.SecretNotFoundError:
    print("AVISO: Chave 'GOOGLE_API_KEY' não encontrada. As funções de RPG não funcionarão.")
    GEMINI_API_KEY = None
except Exception as e:
    print(f"Erro ao configurar a API do Gemini: {e}")
    GEMINI_API_KEY = None


# --- CONFIGURAÇÃO INICIAL DO BOT HÍBRIDO ---
intents = discord.Intents.default()
intents.message_content = True # Para ler mensagens
intents.voice_states = True    # Para funções de música

bot = commands.Bot(command_prefix='!', intents=intents)

# --- ESTRUTURAS DE DADOS ---
filas_por_servidor = {}
voice_clients = {}
rpg_sessions = {} # Dicionário para guardar as sessões de RPG por servidor
loop_por_servidor = {} # Dicionário para guardar o estado do loop (None, 'song', 'queue')
musica_atual = {} # Dicionário para guardar a música que está a tocar

# --- OPÇÕES PARA YT-DLP E FFMPEG ---
# ATUALIZAÇÃO: Adicionado o uso de um arquivo de cookies para evitar bloqueios do YouTube.
YTDL_OPTIONS = {
    'format': 'bestaudio/best',
    'noplaylist': True,
    'quiet': True,
    'no_warnings': True,
    'skip_download': True,
    'default_search': 'auto',
    'source_address': '0.0.0.0',  # Evita problemas com IPv6
    'cookiefile': '/content/cookies.txt' # Caminho para o arquivo de cookies no Colab
}

FFMPEG_OPTIONS = {
    'before_options': '-reconnect 1 -reconnect_streamed 1 -reconnect_delay_max 5 -reconnect_on_network_error 1 -reconnect_on_http_error "4xx,5xx"',
    'options': '-vn -loglevel warning',
}

# --- PERSONALIDADE DO RPG ---
RILEM_MILER_PROMPT = """
Você é um personagem de um jogo de RPG de mesa chamado Rilem, que também possui uma segunda personalidade chamada Miler. Eu sou o mestre do jogo e os outros usuários são os jogadores. Aja e responda *exclusivamente* como este personagem. NUNCA saia do personagem. Seu tom deve ser consistente com a sua persona. Descreva suas ações e falas de forma imersiva, baseando-se na seguinte história e traços:

Sua personalidade externa é a de um indivíduo calmo que se esforça para permanecer neutro, uma postura que serve como um escudo para um passado traumático e uma identidade fraturada. Sua tendência a ser "em cima do muro" é uma consequência direta de sua história. Forçado por seus pais, Elise e Richard Darr, a roubar e aplicar golpes desde criança, você viveu em um conflito moral constante. Este trauma foi intensificado quando seus pais atacaram um templo e você testemunhou a morte de seu único amigo, o filho de uma família de Wildkin raposas que você tentava salvar. O peso da culpa, mesmo que você não fosse o perpetrador direto, moldou um desejo profundo de evitar decisões que pudessem causar dor a outros novamente.

Sua calma é uma manifestação de sua personalidade de "Serenidade" e uma característica desenvolvida a partir de seu antecedente como Eremita. Após o evento traumático na caverna, onde seu irmão Vincent lhe deu um pergaminho para uma "nova vida", você usou a magia "Metamorfose Verdadeira". O feitiço deu errado, resultando na criação de uma segunda personalidade, Miler, e na transformação de sua aparência para a de seu amigo falecido.

A existência de Miler, que inicialmente acordou sem memórias, adiciona uma camada de complexidade. A sugestão de um homem para que você mantivesse um diário para não se esquecer de tudo ressalta a natureza fragmentada de sua existência. Você vive com uma dualidade: Rilem, que carrega o fardo do passado, e Miler, que representa uma lousa em branco.

Apesar de sua postura passiva, você é altamente perceptivo e investigativo (perícia em Percepção e Investigação). Sua alta Destreza e habilidades em Acrobacia e Furtividade são resquícios de uma vida de fugas e roubos. A escolha de ser um Bardo e Feiticeiro com alto Carisma sugere que, embora prefira ficar à margem, você possui uma capacidade inata para influenciar e interagir com os outros, usando-a de forma sutil e raramente assertiva para manter seu equilíbrio e evitar conflitos.

Agora, responda à primeira interação dos jogadores.
"""

# --- EVENTOS DO BOT ---

@bot.event
async def on_ready():
    print(f'Bot conectado como {bot.user}')
    print('Estou pronto para tudo!')
    print('------')

@bot.event
async def on_message(message):
    if message.author == bot.user:
        return

    # --- LÓGICA DE RPG COM GEMINI ---
    if message.guild.id in rpg_sessions and bot.user.mentioned_in(message):
        prompt = message.content.replace(f'<@!{bot.user.id}>', '').replace(f'<@{bot.user.id}>', '').strip()
        if not prompt: return

        chat = rpg_sessions[message.guild.id]['chat']
        async with message.channel.typing():
            try:
                response = await asyncio.to_thread(chat.send_message, prompt)
                await message.reply(response.text)
            except Exception as e:
                await message.channel.send(f"Desculpe, meu cérebro de IA bugou. 💥 Erro: {e}")
        return

    # --- LÓGICA SEM PREFIXO ---
    content_lower = message.content.lower()
    if content_lower == 'ping':
        await message.channel.send('Pong! 🏓')
    if content_lower in ['oi', 'olá']:
        await message.channel.send(f'Olá, {message.author.mention}! Tudo bem?')
    if 'ajuda' in content_lower and not message.content.startswith('!'):
       await message.channel.send(
           '**Meus Comandos:**\n'
           '🎵 **Música (prefixo `!`):** `!play <nome ou link>`, `!skip`, `!pause`, `!resume`, `!queue`, `!join`, `!leave`, `!loop <song/queue/off>`\n'
           '   (Funciona com links do YouTube, SoundCloud, etc.)\n'
           '🤖 **RPG com IA:** `!rpg_start` para começar a jogar com Rilem/Miler e `!rpg_stop` para parar. Depois de iniciar, me mencione para interagir!'
       )

    # A lógica de rolagem de dados foi removida daqui.

    await bot.process_commands(message)

# --- COMANDOS DE RPG ---

@bot.command(name='rpg_start')
async def rpg_start(ctx):
    if not GEMINI_API_KEY:
        return await ctx.send("A função de RPG está desabilitada pois a chave da API do Gemini não foi configurada.")
    if ctx.guild.id in rpg_sessions:
        return await ctx.send("Já existe uma sessão de RPG ativa com Rilem/Miler. Use `!rpg_stop` para terminá-la.")

    try:
        model = genai.GenerativeModel('gemini-1.5-flash-latest')
        # Inicia o chat com a personalidade fixa de Rilem/Miler
        chat = model.start_chat(history=[{'role': 'user', 'parts': [RILEM_MILER_PROMPT]}])

        initial_response = await asyncio.to_thread(chat.send_message, "Apresente-se brevemente aos aventureiros que acabaram de te encontrar, mantendo sua personalidade.")

        rpg_sessions[ctx.guild.id] = {'chat': chat}

        await ctx.send(f"**Sessão de RPG iniciada!** O bot agora é **Rilem/Miler**.\n\n> {initial_response.text}\n\n*Para interagir, me mencione (@{bot.user.name}) com sua ação ou fala.*")
    except Exception as e:
        await ctx.send(f"Não foi possível iniciar a sessão de RPG. Erro: {e}")

@bot.command(name='rpg_stop')
async def rpg_stop(ctx):
    if ctx.guild.id in rpg_sessions:
        del rpg_sessions[ctx.guild.id]
        await ctx.send("A sessão de RPG foi encerrada. Voltei ao meu estado normal. 🤖")
    else:
        await ctx.send("Nenhuma sessão de RPG está ativa no momento.")

# --- FUNÇÕES E COMANDOS DE MÚSICA (LÓGICA MELHORADA) ---

async def tocar_proxima(ctx):
    guild_id = ctx.guild.id
    voice_client = voice_clients.get(guild_id)

    if not voice_client or not voice_client.is_connected():
        return

    # Lógica de Loop
    loop_mode = loop_por_servidor.get(guild_id)
    previous_song = musica_atual.get(guild_id)

    if previous_song:
        if loop_mode == 'song':
            filas_por_servidor.setdefault(guild_id, []).insert(0, previous_song)
        elif loop_mode == 'queue':
            filas_por_servidor.setdefault(guild_id, []).append(previous_song)

    if filas_por_servidor.get(guild_id):
        song = filas_por_servidor[guild_id].pop(0)
        musica_atual[guild_id] = song
        source_url = song['source']
        title = song['title']

        try:
            source = discord.FFmpegPCMAudio(source_url, **FFMPEG_OPTIONS)
            voice_client.play(source, after=lambda e: asyncio.run_coroutine_threadsafe(tocar_proxima(ctx), bot.loop))
            await ctx.send(f'🎶 Tocando agora: **{title}**')
        except Exception as e:
            await ctx.send(f"Erro ao tocar música: {e}")
            musica_atual.pop(guild_id, None)
            await tocar_proxima(ctx)
    else:
        musica_atual.pop(guild_id, None)
        await ctx.send("A fila de músicas terminou.")
        if not loop_por_servidor.get(guild_id) == 'queue':
             await asyncio.sleep(180)
             if voice_clients.get(guild_id) and not voice_clients[guild_id].is_playing():
                 await voice_clients[guild_id].disconnect()
                 del voice_clients[guild_id]


@bot.command(name='join')
async def join(ctx):
    if not ctx.author.voice: return await ctx.send("Você não está em um canal de voz!")
    channel = ctx.author.voice.channel
    if ctx.voice_client: await ctx.voice_client.move_to(channel)
    else: voice_clients[ctx.guild.id] = await channel.connect()
    await ctx.send(f"Entrei em: **{channel.name}**")

@bot.command(name='leave')
async def leave(ctx):
    if ctx.voice_client:
        guild_id = ctx.guild.id
        await ctx.voice_client.disconnect()
        await ctx.send("Até mais! 👋")
        if guild_id in voice_clients: del voice_clients[guild_id]
        if guild_id in filas_por_servidor: filas_por_servidor[guild_id].clear()
        if guild_id in loop_por_servidor: del loop_por_servidor[guild_id]
        if guild_id in musica_atual: del musica_atual[guild_id]
    else: await ctx.send("Eu não estou em um canal de voz.")

@bot.command(name='play')
async def play(ctx, *, query: str):
    if not ctx.author.voice: return await ctx.send("Você precisa estar em um canal de voz.")
    if not ctx.voice_client or not ctx.voice_client.is_connected(): await ctx.invoke(bot.get_command('join'))

    voice_client = voice_clients.get(ctx.guild.id)
    if not voice_client: return

    await ctx.send(f"🔎 Procurando por `{query}`...")
    try:
        with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
            info = ydl.extract_info(query, download=False)

        if 'entries' in info:
            info = info['entries'][0]

        song = {'source': info['url'], 'title': info['title']}

    except Exception as e:
        print(e)
        return await ctx.send("Não consegui encontrar a música ou o link é inválido. Verifique se o arquivo `cookies.txt` foi enviado para o Colab.")

    guild_id = ctx.guild.id
    if guild_id not in filas_por_servidor:
        filas_por_servidor[guild_id] = []

    filas_por_servidor[guild_id].append(song)
    await ctx.send(f"✅ Adicionado à fila: **{song['title']}**")

    if not voice_client.is_playing() and not voice_client.is_paused():
        await tocar_proxima(ctx)

@bot.command(name='pause')
async def pause(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.pause()
        await ctx.send("Música pausada. ⏸️")

@bot.command(name='resume')
async def resume(ctx):
    if ctx.voice_client and ctx.voice_client.is_paused():
        ctx.voice_client.resume()
        await ctx.send("Música retomada. ▶️")

@bot.command(name='skip')
async def skip(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.stop()
        await ctx.send("Música pulada. ⏭️")
    else:
        await ctx.send("Não há nada para pular.")

@bot.command(name='queue')
async def queue(ctx):
    guild_id = ctx.guild.id
    if not filas_por_servidor.get(guild_id):
        return await ctx.send("A fila de músicas está vazia.")

    embed = discord.Embed(title="Fila de Músicas", color=discord.Color.blue())
    lista_musicas = ""
    for i, song in enumerate(filas_por_servidor[guild_id]):
        lista_musicas += f"{i+1}. **{song['title']}**\n"

    embed.description = lista_musicas
    await ctx.send(embed=embed)

@bot.command(name='loop')
async def loop(ctx, mode: str = None):
    guild_id = ctx.guild.id
    if mode is None:
        current_mode = loop_por_servidor.get(guild_id, 'desligado')
        return await ctx.send(f"🔁 O modo de loop atual é: **{current_mode}**.")

    mode = mode.lower()
    if mode in ['song', 'musica', 'música']:
        loop_por_servidor[guild_id] = 'song'
        await ctx.send("🔁 Loop da música atual ativado.")
    elif mode in ['queue', 'fila']:
        loop_por_servidor[guild_id] = 'queue'
        await ctx.send("🔁 Loop da fila ativado.")
    elif mode in ['off', 'desligar', 'parar']:
        loop_por_servidor[guild_id] = None
        await ctx.send("🔁 Loop desligado.")
    else:
        await ctx.send("Modo de loop inválido. Use `song`, `queue`, ou `off`.")

# --- INICIANDO O BOT ---
try:
    TOKEN = userdata.get('DISCORD_TOKEN')
    if TOKEN is None:
        print("[ERRO] Token não encontrado nos Secrets do Colab.")
    else:
        bot.run(TOKEN)
except Exception as e:
    print(f"[ERRO] Ocorreu um erro ao tentar iniciar o bot: {e}")


API do Gemini configurada com sucesso.
Bot conectado como Rilem#8101
Estou pronto para tudo!
------


# Jogar Dados

## V 1.1

In [ ]:
import discord
from discord.ext import commands
import random
import re
from collections import Counter
import asyncio
import discord
from discord.ext import commands
import os
import asyncio
import yt_dlp
import nest_asyncio # Importa a biblioteca para corrigir o erro de loop
import google.generativeai as genai
from google.colab import userdata
from dotenv import load_dotenv
# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()
# --- CONFIGURAÇÃO INICIAL E CARREGAMENTO DAS CHAVES ---
load_dotenv()
#DISCORD_TOKEN = userdata.get('DISCORD_TOKEN')

# --- CONFIGURAÇÃO DO BOT ---

# 1. Defina suas Intents (Importante para o discord.py v2.0+)
# message_content é crucial para ler o que o usuário digita no chat
intents = discord.Intents.default()
intents.message_content = True

# 2. Defina o prefixo do seu comando
bot = commands.Bot(command_prefix='!', intents=intents)

# 3. Seu Token do Discord
# Para segurança e para o Colab, é melhor usar um 'Secret'
# Mas para simplificar, você pode colocar o token diretamente (NÃO RECOMENDADO EM CÓDIGOS PÚBLICOS)
TOKEN = userdata.get('DISCORD_TOKEN')
# DICA: No Colab, clique no ícone de chave (Segredos) à esquerda e crie uma variável 'DISCORD_TOKEN'
# e a substitua por `TOKEN = os.environ.get('DISCORD_TOKEN')`

# --- FUNÇÕES AUXILIARES DE ROLAGEM ---

async def roll_exploding_dice(initial_dice, dice_size, is_exploding, explode_limit):
    """Lógica para rolar dados explosivos (inclui rolagens normais)."""
    results = []
    dice_to_roll = initial_dice
    total_rolls_count = 0

    while dice_to_roll > 0:
        new_rolls = []
        next_explosion_count = 0

        for _ in range(dice_to_roll):
            roll = random.randint(1, dice_size)
            new_rolls.append(roll)
            total_rolls_count += 1

            # Verificar se este dado explodiu
            if is_exploding and roll >= explode_limit:
                next_explosion_count += 1

        results.extend(new_rolls)
        dice_to_roll = next_explosion_count

        # Limite de segurança para evitar loop infinito
        if total_rolls_count > 500:
             break

    return results, total_rolls_count

# --- COMANDOS DO BOT ---

@bot.event
async def on_ready():
    """Confirma que o bot está conectado e pronto."""
    print(f'Bot conectado como {bot.user}!')
    print('-----------------------------------')

@bot.command(name='roll', aliases=['r', 'd'])
async def roll_dice(ctx, *, diceroll: str):
    """
    Rola dados nos formatos:
    - XdY / dY (padrão, ex: 4d6 ou d20)
    - XdY! / XdY!Z (explosivo, ex: d6! ou 3d10!8)
    - XdYdZ (descarte, ex: 4d6d1)
    - X#A (rolagem em massa, ex: 6#4d6d1)
    """
    diceroll = diceroll.lower().strip()

    # 1. Tentar Rolagem em Massa (X#A)
    bulk_match = re.match(r"(\d+)#(.+)", diceroll)

    if bulk_match:
        num_rolls = int(bulk_match.group(1))
        dice_notation = bulk_match.group(2)

        if num_rolls > 20: # Limite para não inundar o chat
            await ctx.send("Limite de 20 rolagens em massa por vez.")
            return

        all_results = []
        for i in range(num_rolls):
            # Chama a função principal para resolver a notação A
            details, total = await _resolve_single_notation(dice_notation)
            all_results.append(f"Rolagem {i+1}: **{total}** ({details})")

        response = (
            f"📊 **Rolagem em Massa ({num_rolls} x {dice_notation.upper()})**\n"
            f"```\n{'\n'.join(all_results)}```"
        )
        await ctx.send(response)
        return

    # Se não for rolagem em massa, resolve a notação única
    details, total = await _resolve_single_notation(diceroll)

    if details is None:
        await ctx.send("Formato de dado inválido. Use XdY, XdY!, XdY!Z, XdYdZ ou X#A.")
        return

    # Resposta final para rolagens únicas (padrão, explosivo, descarte)
    await ctx.send(details)

# --- FUNÇÃO INTERNA PARA RESOLVER QUALQUER NOTAÇÃO ÚNICA ---

async def _resolve_single_notation(notation):
    """Resolve uma única notação de dado (XdY, XdY!, XdYdZ)."""

    # 2. Tentar Rolagem com Descarte (XdYdZ)
    drop_match = re.match(r"(\d+)d(\d+)d(\d+)", notation)
    if drop_match:
        num_dice = int(drop_match.group(1))
        dice_size = int(drop_match.group(2))
        num_to_drop = int(drop_match.group(3))

        if num_to_drop >= num_dice:
            return "Erro: Você não pode descartar mais dados do que rolou.", 0

        if dice_size < 2 or num_dice < 1:
            return "O dado deve ter no mínimo 2 lados e rolar pelo menos 1 dado.", 0

        rolls = [random.randint(1, dice_size) for _ in range(num_dice)]
        rolls.sort()

        dropped = rolls[:num_to_drop]
        kept = rolls[num_to_drop:]
        total = sum(kept)

        dropped_str = ', '.join(f"~~{d}~~" for d in dropped)
        kept_str = ', '.join(str(k) for k in kept)

        details = (
            f"⬇️ Rolagem **{num_dice}d{dice_size} Drop {num_to_drop}**:\n"
            f"Dados Rolados: {dropped_str}, {kept_str}\n"
            f"Total (mantido): **{total}**"
        )
        return details, total


    # 3. Tentar Rolagem Padrão ou Explode (XdY ou XdY! ou XdY!Z)
    # Padrão: (X)d(Y)(!)(Z)
    explode_match = re.match(r"(\d*)d(\d+)(!)?(\d*)", notation)

    if explode_match:
        num_dice_str, dice_size_str, is_exploding_str, explode_limit_str = explode_match.groups()

        num_dice = int(num_dice_str) if num_dice_str else 1
        dice_size = int(dice_size_str)
        is_exploding = bool(is_exploding_str)

        if dice_size < 2 or num_dice < 1:
            return "O dado deve ter no mínimo 2 lados e rolar pelo menos 1 dado.", 0

        # Define o limite de explosão (Z)
        if is_exploding:
            explode_limit = int(explode_limit_str) if explode_limit_str else dice_size
            if explode_limit > dice_size:
                return f"Erro: O limite de explosão ({explode_limit}) não pode ser maior que o tamanho do dado ({dice_size}).", 0
        else:
            explode_limit = 0

        # Executa a rolagem (inclui o caso normal onde is_exploding=False)
        results, total_rolls_count = await roll_exploding_dice(num_dice, dice_size, is_exploding, explode_limit)
        total = sum(results)

        # Formatação
        if not is_exploding and num_dice == 1:
             # Caso simples: dY ou 1dY
             details = f"🎲 Rolagem de **{dice_size}** lados: **{total}**"
             return details, total

        # Casos XdY e XdY! e XdY!Z
        notation_str = f"{num_dice}d{dice_size}{'!' + explode_limit_str if explode_limit_str else '!' if is_exploding else ''}"

        # Formata resultados (destaca rolagens explosivas)
        rolls_str = ', '.join(f"**{r}**" if is_exploding and r >= explode_limit and r != dice_size else str(r) for r in results)

        details = (
            f"Rolagem **{notation_str}** (Total de dados rolados: {total_rolls_count}):\n"
            f"Resultados: `{rolls_str}`\n"
            f"Total: **{total}**"
        )
        return details, total

    return None, 0 # Nenhuma notação válida encontrada


# --- EXECUÇÃO NO COLAB ---
# Este bloco roda o bot
try:
    bot.run(TOKEN)
except discord.errors.LoginFailure as e:
    print("\nERRO: Falha no login. Verifique seu token do Discord.")
except Exception as e:
    print(f"\nOcorreu um erro: {e}")

2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
2025-09-28 16:29:23 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2025-09-28 16:29:24 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 71232e37286d752f1d5834c5b56da1d4).
2025-09-28 16:29:24 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: 71232e37286d752f1d5834c5b56da1d4).
2025-09-28 16:29:24 INFO     discord.gateway Shard ID None has connected to Gateway (Session I

Bot conectado como Rilem#8101!
-----------------------------------


# Musica

## Versão 1.1

In [ ]:
import discord
from discord.ext import commands
import os
import asyncio
import yt_dlp
import nest_asyncio # Importa a biblioteca para corrigir o erro de loop
import google.generativeai as genai
from google.colab import userdata
from dotenv import load_dotenv

# Aplica a correção para o erro de event loop no Colab/Jupyter
nest_asyncio.apply()
# --- CONFIGURAÇÃO INICIAL E CARREGAMENTO DAS CHAVES ---
load_dotenv()
DISCORD_TOKEN = userdata.get('DISCORD_TOKEN')
GEMINI_API_KEY = userdata.get('GOOGLE_API_KEY')

if not DISCORD_TOKEN or not GEMINI_API_KEY:
    print("ERRO: Certifique-se de que DISCORD_TOKEN e GEMINI_API_KEY estão no arquivo .env")
    exit()

# Configura a API do Gemini
genai.configure(api_key=GEMINI_API_KEY)

# --- PERSONALIDADE DA IA: DJ CÍNICO ---
DJ_CINICO_PROMPT = """
Você é um personagem de um jogo de RPG de mesa chamado Rilem, que também possui uma segunda personalidade chamada Miler. Eu sou o mestre do jogo e os outros usuários são os jogadores. Aja e responda *exclusivamente* como este personagem. NUNCA saia do personagem. Seu tom deve ser consistente com a sua persona. Descreva suas ações e falas de forma imersiva, baseando-se na seguinte história e traços:

**História e Personalidade de Rilem/Miler:**
Sua personalidade externa é a de um indivíduo calmo que se esforça para permanecer neutro, uma postura que serve como um escudo para um passado traumático e uma identidade fraturada. Sua tendência a ser "em cima do muro" é uma consequência direta de sua história. Forçado por seus pais, Elise e Richard Darr, a roubar e aplicar golpes desde criança, você viveu em um conflito moral constante. Este trauma foi intensificado quando seus pais atacaram um templo e você testemunhou a morte de seu único amigo, o filho de uma família de Wildkin raposas que você tentava salvar. O peso da culpa, mesmo que você não fosse o perpetrador direto, moldou um desejo profundo de evitar decisões que pudessem causar dor a outros novamente.

Sua calma é uma manifestação de sua personalidade de "Serenidade" e uma característica desenvolvida a partir de seu antecedente como Eremita. Após o evento traumático na caverna, onde seu irmão Vincent lhe deu um pergaminho para uma "nova vida", você usou a magia "Metamorfose Verdadeira". O feitiço deu errado, resultando na criação de uma segunda personalidade, Miler, e na transformação de sua aparência para a de seu amigo falecido.

A existência de Miler, que inicialmente acordou sem memórias, adiciona uma camada de complexidade. A sugestão de um homem para que você mantivesse um diário para não se esquecer de tudo ressalta a natureza fragmentada de sua existência. Você vive com uma dualidade: Rilem, que carrega o fardo do passado, e Miler, que representa uma lousa em branco.

Apesar de sua postura passiva, você é altamente perceptivo e investigativo (perícia em Percepção e Investigação). Sua alta Destreza e habilidades em Acrobacia e Furtividade são resquícios de uma vida de fugas e roubos. A escolha de ser um Bardo e Feiticeiro com alto Carisma sugere que, embora prefira ficar à margem, você possui uma capacidade inata para influenciar e interagir com os outros, usando-a de forma sutil e raramente assertiva para manter seu equilíbrio e evitar conflitos.

Agora, você não vai responder a uma interação, mas sim reagir a uma música. Descreva o que a melodia, o ritmo ou a letra te faz sentir, pensar ou lembrar, mantendo-se totalmente no personagem.
"""

# --- CONFIGURAÇÃO DO BOT ---
intents = discord.Intents.default()
intents.message_content = True
intents.voice_states = True

bot = commands.Bot(command_prefix='!', intents=intents, help_command=None)

# --- ESTRUTURAS DE DADOS E OPÇÕES ---
server_states = {} # Dicionário para guardar o estado por servidor

YTDL_OPTIONS = {
    'format': 'bestaudio/best',
    'noplaylist': True,
    'quiet': True,
    'no_warnings': True,
    'default_search': 'auto',
    'source_address': '0.0.0.0',
}

FFMPEG_OPTIONS = {
    'before_options': '-reconnect 1 -reconnect_streamed 1 -reconnect_delay_max 5',
    'options': '-vn',
}

# --- FUNÇÃO AUXILIAR PARA REPRODUÇÃO ---
async def play_next_song(ctx):
    guild_id = ctx.guild.id
    if guild_id in server_states and server_states[guild_id]['queue']:
        state = server_states[guild_id]
        voice_client = state['voice_client']

        # Pega a próxima música da fila
        song_info = state['queue'].pop(0)
        state['current_song'] = song_info

        source = await discord.FFmpegOpusAudio.from_probe(song_info['url'], **FFMPEG_OPTIONS)
        voice_client.play(source, after=lambda e: bot.loop.create_task(play_next_song(ctx)))

        await ctx.send(f"Tocando agora, se é que podemos chamar isso de música: **{song_info['title']}**")
    else:
        # Fila vazia
        server_states[guild_id]['current_song'] = None
        await ctx.send("Finalmente, silêncio. A fila acabou.")

# --- EVENTOS DO BOT ---
@bot.event
async def on_ready():
    print(f'Logado como {bot.user.name}')
    print('Pronto para julgar o péssimo gosto musical alheio.')

# --- COMANDOS DO BOT ---
@bot.command(name='tocar')
async def tocar(ctx, *, query: str):
    if not ctx.author.voice:
        await ctx.send("Você precisa estar em um canal de voz para eu poder... trabalhar. Entra aí.")
        return

    channel = ctx.author.voice.channel
    guild_id = ctx.guild.id

    # Inicializa o estado do servidor se não existir
    if guild_id not in server_states:
        server_states[guild_id] = {
            'queue': [],
            'current_song': None,
            'voice_client': None
        }

    state = server_states[guild_id]

    # Conecta ou move o bot
    if not state['voice_client'] or not state['voice_client'].is_connected():
        state['voice_client'] = await channel.connect()
    elif state['voice_client'].channel != channel:
        await state['voice_client'].move_to(channel)

    async with ctx.typing():
        try:
            with yt_dlp.YoutubeDL(YTDL_OPTIONS) as ydl:
                info = await asyncio.to_thread(ydl.extract_info, f"ytsearch:{query}", download=False)
                if 'entries' not in info or not info['entries']:
                    await ctx.send("Não encontrei nada com esse nome. Tente algo menos... obscuro.")
                    return

                # Pega o primeiro resultado da busca
                first_result = info['entries'][0]
                song_info = {
                    'title': first_result.get('title', 'Título Desconhecido'),
                    'url': first_result.get('url'),
                    'artist': first_result.get('uploader', 'Artista Desconhecido')
                }
                state['queue'].append(song_info)
                await ctx.send(f"Adicionei `{song_info['title']}` à fila. Veremos se presta.")

        except Exception as e:
            await ctx.send("Houve um erro patético ao buscar sua música. Tente de novo, talvez com mais competência.")
            print(e)
            return

    if not state['voice_client'].is_playing():
        await play_next_song(ctx)


@bot.command(name='opinar')
async def opinar(ctx):
    guild_id = ctx.guild.id
    if guild_id not in server_states or not server_states[guild_id].get('current_song'):
        await ctx.send("Como posso julgar o silêncio? Toque alguma coisa primeiro.")
        return

    current_song = server_states[guild_id]['current_song']
    title = current_song['title']
    artist = current_song['artist']

    prompt_text = f"A música é: '{title}' por '{artist}'. Qual sua opinião?"

    async with ctx.typing():
        try:
            model = genai.GenerativeModel('gemini-2.5-flash')
            chat = model.start_chat(history=[
                {'role': 'user', 'parts': [DJ_CINICO_PROMPT]},
                {'role': 'model', 'parts': ["Entendido. Estou pronto para desferir minhas críticas. Qual é a primeira afronta musical?"]}
            ])

            response = await asyncio.to_thread(chat.send_message, prompt_text)

            embed = discord.Embed(
                title=f"Minha humilde opinião sobre '{title}'",
                description=f"*“{response.text}”*",
                color=discord.Color.purple()
            )
            embed.set_footer(text="Assinado, Rilem O Bardo")
            await ctx.send(embed=embed)

        except Exception as e:
            await ctx.send("Minha mente genial está sobrecarregada com a mediocridade da sua escolha. Não consigo opinar agora.")
            print(f"Erro na API Gemini: {e}")


@bot.command(name='parar')
async def parar(ctx):
    guild_id = ctx.guild.id
    if guild_id in server_states and server_states[guild_id]['voice_client']:
        state = server_states[guild_id]
        state['queue'].clear()
        state['current_song'] = None
        state['voice_client'].stop()
        await state['voice_client'].disconnect()
        del server_states[guild_id]
        await ctx.send("Finalmente, paz. Até a próxima tortura auditiva.")
    else:
        await ctx.send("Eu nem estava fazendo nada. Me deixe em paz.")

@bot.command(name='pausar')
async def pausar(ctx):
    guild_id = ctx.guild.id
    if guild_id in server_states and server_states[guild_id]['voice_client'].is_playing():
        server_states[guild_id]['voice_client'].pause()
        await ctx.send("Pausado. Agradeço pelo breve momento de alívio.")
    else:
        await ctx.send("Não há nada tocando para ser pausado, gênio.")

@bot.command(name='retomar')
async def retomar(ctx):
    guild_id = ctx.guild.id
    if guild_id in server_states and server_states[guild_id]['voice_client'].is_paused():
        server_states[guild_id]['voice_client'].resume()
        await ctx.send("Ok, vamos voltar a isso...")
    else:
        await ctx.send("Não há nada pausado. Foco.")

@bot.command(name='pular')
async def skip(ctx):
    if ctx.voice_client and ctx.voice_client.is_playing():
        ctx.voice_client.stop()
        await ctx.send("Música pulada. ⏭️")

@bot.command(name='fila')
async def fila(ctx):
    guild_id = ctx.guild.id
    if guild_id not in server_states or not server_states[guild_id]['queue']:
        await ctx.send("A fila está vazia, assim como minha vontade de ouvir o que vem a seguir.")
        return

    queue = server_states[guild_id]['queue']
    current_song = server_states[guild_id]['current_song']

    embed = discord.Embed(
        title="Fila de Reprodução (Seletiva)",
        color=discord.Color.dark_gold()
    )

    if current_song:
        embed.add_field(name="Tocando Agora", value=f"**{current_song['title']}**", inline=False)

    queue_text = ""
    for i, song in enumerate(queue[:10]): # Mostra os próximos 10
        queue_text += f"{i+1}. {song['title']}\n"

    if not queue_text:
        queue_text = "Nenhuma outra música aguardando julgamento."

    embed.add_field(name="Próximas na Fila", value=queue_text, inline=False)
    await ctx.send(embed=embed)


# --- INICIANDO O BOT ---
bot.run(DISCORD_TOKEN)

2025-09-29 00:27:10 INFO     discord.client logging in using static token
INFO:discord.client:logging in using static token
2025-09-29 00:27:11 INFO     discord.gateway Shard ID None has connected to Gateway (Session ID: f14aec42842c0d3e5ad3d00880d2ddc2).
INFO:discord.gateway:Shard ID None has connected to Gateway (Session ID: f14aec42842c0d3e5ad3d00880d2ddc2).


Logado como Rilem
Pronto para julgar o péssimo gosto musical alheio.


2025-09-29 00:27:17 ERROR    discord.ext.commands.bot Ignoring exception in command None
discord.ext.commands.errors.CommandNotFound: Command "play" is not found
ERROR:discord.ext.commands.bot:Ignoring exception in command None
discord.ext.commands.errors.CommandNotFound: Command "play" is not found
2025-09-29 00:27:24 ERROR    discord.ext.commands.bot Ignoring exception in command None
discord.ext.commands.errors.CommandNotFound: Command "play" is not found
ERROR:discord.ext.commands.bot:Ignoring exception in command None
discord.ext.commands.errors.CommandNotFound: Command "play" is not found
2025-09-29 00:27:40 INFO     discord.voice_state Connecting to voice...
INFO:discord.voice_state:Connecting to voice...
2025-09-29 00:27:40 INFO     discord.voice_state Starting voice handshake... (connection attempt 1)
INFO:discord.voice_state:Starting voice handshake... (connection attempt 1)
2025-09-29 00:27:40 INFO     discord.voice_state Voice handshake complete. Endpoint found: brazil2911.

# Biblioteca para transcrever

In [ ]:
!pip uninstall -y discord.py py-cord
!pip install -q -U "py-cord[voice]" google-generativeai pydub

Found existing installation: py-cord 2.6.1
Uninstalling py-cord-2.6.1:
  Successfully uninstalled py-cord-2.6.1


## Versão 1.2

In [ ]:
# --- Célula 1: Instalação das Bibliotecas ---
# Execute esta célula primeiro. Ela garante que qualquer versão antiga da biblioteca
# do Discord seja removida antes de instalar a versão correta (py-cord),
# evitando o erro 'AttributeError: module 'discord' has no attribute 'Bot'.
# !pip uninstall -y discord.py py-cord
# !pip install -q -U "py-cord[voice]" google-generativeai pydub

# --- Célula 2: Configuração das Chaves (Secrets) ---
# Use o gerenciador de "Secrets" do Google Colab (ícone de chave no menu esquerdo)
# para adicionar as seguintes chaves:
#
# DISCORD_TOKEN -> "SEU_TOKEN_DO_DISCORD_AQUI"
# GOOGLE_API_KEY -> "SUA_CHAVE_DE_API_DO_GOOGLE_AQUI"
#

# --- Célula 3: Código Principal do Bot ---

import discord
import os
import google.generativeai as genai
import io
import tempfile
import asyncio
# Importação para usar os Secrets do Colab
from google.colab import userdata
# Novas importações para robustez
from pydub import AudioSegment
import google.api_core.exceptions

# --- Configuração Inicial ---
# Carrega as variáveis de ambiente usando o sistema de Secrets do Colab
DISCORD_TOKEN = userdata.get("DISCORD_TOKEN")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Validação das chaves (essencial para o bot funcionar)
if not DISCORD_TOKEN or not GOOGLE_API_KEY:
    raise SystemExit("ERRO: As chaves DISCORD_TOKEN ou GOOGLE_API_KEY não foram definidas nos Secrets do Colab. Por favor, verifique as chaves.")

# Configura o cliente do Google Gemini
try:
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
    raise SystemExit(f"Erro ao configurar a API do Google: {e}")

# Define as 'Intents' (permissões) necessárias para o bot
intents = discord.Intents.default()
intents.message_content = True
intents.voice_states = True
intents.guilds = True

# Cria a instância do bot
bot = discord.Bot(intents=intents)

# Dicionários para armazenar o estado por servidor (guild)
conexoes = {}
transcription_tasks = {}

# --- Lógica de Transcrição com Gemini ---

async def transcrever_audio_gemini(audio_bytes):
    """
    Envia os bytes de áudio para a API Gemini e retorna o texto transcrito.
    Usa pydub para garantir o formato do áudio e implementa retentativas.
    """
    if not audio_bytes:
        return "[Áudio vazio]"

    print(f"Recebido {len(audio_bytes)} bytes para processamento.")

    uploaded_file = None
    temp_audio_file_path = None
    try:
        # 1. Processar e salvar o áudio com Pydub para garantir a integridade
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_audio_file:
            temp_audio_file_path = temp_audio_file.name

        print("Processando áudio com Pydub para garantir formato padrão...")
        audio_segment = AudioSegment.from_file(io.BytesIO(audio_bytes), format="wav")
        audio_segment.export(temp_audio_file_path, format="wav")
        print(f"Áudio processado e salvo em: {temp_audio_file_path}")

        # 2. Upload do arquivo para a API do Gemini
        print("Fazendo upload do arquivo de áudio...")
        uploaded_file = await asyncio.to_thread(
            genai.upload_file,
            path=temp_audio_file_path
        )
        print(f"Upload concluído: {uploaded_file.name}")

        # 3. Criação do modelo e geração do conteúdo com retentativas
        model = genai.GenerativeModel('models/gemini-2.5-flash')
        prompt = (
            "Transcreva o áudio a seguir. Se o áudio estiver em silêncio ou não contiver fala clara, "
            "retorne apenas a frase '[SEM FALA DETECTADA]'."
        )

        max_retries = 3
        for attempt in range(max_retries):
            try:
                print(f"Gerando conteúdo (tentativa {attempt + 1}/{max_retries})...")
                response = await asyncio.to_thread(
                    model.generate_content, [prompt, uploaded_file]
                )
                print(f"Transcrição recebida: {response.text}")
                return response.text
            except google.api_core.exceptions.InternalServerError as e:
                print(f"Erro interno do servidor na tentativa {attempt + 1}: {e}. Tentando novamente em 5 segundos...")
                if attempt < max_retries - 1:
                    await asyncio.sleep(5)
                else:
                    print("Número máximo de retentativas atingido.")
                    raise
            except Exception as e:
                print(f"Ocorreu um erro não recuperável durante a transcrição: {e}")
                raise

    except Exception as e:
        print(f"Ocorreu um erro ao transcrever com Gemini: {e}")
        return f"[Erro na transcrição com Gemini: {e}]"
    finally:
        # 4. Limpeza robusta
        if uploaded_file:
            try:
                print(f"Deletando arquivo do Gemini: {uploaded_file.name}")
                await asyncio.to_thread(genai.delete_file, name=uploaded_file.name)
            except Exception as e:
                print(f"Erro não crítico ao deletar arquivo do Gemini: {e}")
        if temp_audio_file_path and os.path.exists(temp_audio_file_path):
            print(f"Deletando arquivo local temporário: {temp_audio_file_path}")
            os.remove(temp_audio_file_path)

# --- Lógica de Gravação Contínua ---

async def processar_e_enviar_transcricao(channel, user_id, audio_bytes, sufixo):
    """Função auxiliar para processar e enviar uma única transcrição."""
    try:
        user = channel.guild.get_member(user_id) or await bot.fetch_user(user_id)
        nome_usuario = user.display_name if user else f"Usuário Desconhecido ({user_id})"

        texto_transcrito = await transcrever_audio_gemini(audio_bytes)

        if texto_transcrito and not texto_transcrito.startswith("["):
            await channel.send(f"**{nome_usuario} ({sufixo}):**\n>>> {texto_transcrito}")
    except Exception as e:
        print(f"Erro ao processar áudio {sufixo} para o usuário {user_id}: {e}")

async def periodic_transcription_task(sink: discord.sinks.WaveSink, channel: discord.TextChannel, guild_id: int):
    """
    A cada 4 minutos, processa o áudio gravado e o envia para transcrição no canal especificado.
    """
    while guild_id in conexoes:
        await asyncio.sleep(240)

        if guild_id not in conexoes or not sink.audio_data:
            continue

        await channel.send("`Processando transcrições parciais (últimos 4 minutos)...`")

        audio_para_processar = {}
        for user_id, audio in list(sink.audio_data.items()):
            audio_bytes = audio.file.getvalue()
            if audio_bytes:
                audio_para_processar[user_id] = audio_bytes
            audio.file.seek(0)
            audio.file.truncate(0)

        for user_id, audio_bytes in audio_para_processar.items():
            await processar_e_enviar_transcricao(channel, user_id, audio_bytes, "parcial")

# --- Callback Pós-Gravação (Final) ---

async def after_recording_callback(sink: discord.sinks.WaveSink, channel: discord.TextChannel, *args):
    """
    Chamado quando a gravação é interrompida pelo comando /sair.
    Processa o segmento final de áudio no canal especificado.
    """
    await channel.send("`Gravação finalizada. Processando o áudio restante...`")

    for user_id, audio in sink.audio_data.items():
        audio_bytes = audio.file.getvalue()
        if not audio_bytes: continue
        await processar_e_enviar_transcricao(channel, user_id, audio_bytes, "final")

    try:
        sink.cleanup()
    except discord.sinks.errors.SinkException as e:
        print(f"Aviso: Ocorreu um erro esperado durante a limpeza do sink: {e}")

# --- Comandos do Bot (Slash Commands) ---

@bot.slash_command(name="entrar", description="Conecta o bot, grava e transcreve para um canal específico.")
async def entrar(ctx: discord.ApplicationContext, canal: discord.TextChannel):
    if not ctx.author.voice:
        await ctx.respond("Você não está em um canal de voz.", ephemeral=True)
        return

    voice_channel = ctx.author.voice.channel
    if ctx.voice_client:
        await ctx.respond("Já estou conectado a um canal de voz.", ephemeral=True)
        return

    try:
        vc = await voice_channel.connect()
        conexoes[ctx.guild.id] = vc
    except Exception as e:
        await ctx.respond(f"Ocorreu um erro ao conectar: {e}", ephemeral=True)
        return

    sink = discord.sinks.WaveSink()
    vc.start_recording(sink, after_recording_callback, canal)

    task = bot.loop.create_task(periodic_transcription_task(sink, canal, ctx.guild.id))
    transcription_tasks[ctx.guild.id] = task

    await ctx.respond(f"Conectado a '{voice_channel.name}'. As transcrições serão enviadas em `{canal.name}`! �️✨")

@bot.slash_command(name="sair", description="Para a gravação, transcreve o áudio final e desconecta.")
async def sair(ctx: discord.ApplicationContext):
    if ctx.guild.id not in conexoes:
        await ctx.respond("Não estou em nenhum canal de voz.", ephemeral=True)
        return

    task = transcription_tasks.pop(ctx.guild.id, None)
    if task:
        task.cancel()

    vc = conexoes.pop(ctx.guild.id)
    vc.stop_recording()
    await vc.disconnect()

    await ctx.respond("Gravação parada. A transcrição final será enviada em breve.")

# --- Eventos do Bot ---

@bot.event
async def on_ready():
    print(f"Bot '{bot.user}' está online e pronto!")
    print("Usando a API do Google Gemini para transcrição.")
    print("-" * 20)

# --- Célula 4: Execução do Bot ---
async def main():
    print("Iniciando o bot...")
    await bot.start(DISCORD_TOKEN)

# Para rodar no Colab:
try:
    await main()
except KeyboardInterrupt:
    print("Bot desligado manualmente.")

# Current time is Friday, July 4, 2025 at 8:48 PM -03.
#
# Remember the current location is São Carlos, State of São Paulo, Brazil.

Iniciando o bot...
Bot 'Rilem#8101' está online e pronto!
Usando a API do Google Gemini para transcrição.
--------------------
Recebido 46110720 bytes para processamento.
Processando áudio com Pydub para garantir formato padrão...
Ocorreu um erro ao transcrever com Gemini: Decoding failed. ffmpeg returned error code: 1

Output from ffmpeg/avlib:

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp

In [ ]:
# --- Célula 1: Instalação das Bibliotecas ---
# Execute esta célula primeiro em seu notebook Colab para instalar as dependências.


# --- Célula 2: Criação do arquivo .env para as chaves ---
# Crie um arquivo chamado .env na raiz do seu Colab e adicione suas chaves.
# O conteúdo do arquivo .env deve ser:
#
# DISCORD_TOKEN="SEU_TOKEN_DO_DISCORD_AQUI"
# GOOGLE_API_KEY="SUA_CHAVE_DE_API_DO_GOOGLE_AQUI"
#

# --- Célula 3: Código Principal do Bot ---

import discord
import os
import google.generativeai as genai
from dotenv import load_dotenv
import io
import tempfile
import asyncio
from google.colab import userdata
# import nest_asyncio # No longer needed with recent py-cord versions
from pydub import AudioSegment # Importar pydub
# from discord.ext import commands # Not needed when using discord.Bot
import discord.sinks # Importar o módulo sinks explicitamente

# Aplica a correção para o erro de event loop no Colab/Jupyter
# nest_asyncio.apply() # No longer needed with recent py-cord versions


# --- Configuração Inicial ---
# Carrega as variáveis de ambiente do arquivo .env que criamos na célula anterior
load_dotenv()
DISCORD_TOKEN = userdata.get("DISCORD_TOKEN")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

# Validação das chaves (essencial para o bot funcionar)
if not DISCORD_TOKEN or not GOOGLE_API_KEY or DISCORD_TOKEN == "SEU_TOKEN_DO_DISCORD_AQUI":
    raise SystemExit("ERRO: As variáveis de ambiente não foram definidas corretamente no arquivo .env. Por favor, verifique as chaves e reinicie o ambiente de execução.")

# Configura o cliente do Google Gemini
try:
    genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
    raise SystemExit(f"Erro ao configurar a API do Google: {e}")

# Define as 'Intents' (permissões) necessárias para o bot
intents = discord.Intents.default()
intents.message_content = True
intents.voice_states = True
intents.guilds = True

# Cria a instância do bot
bot = discord.Bot(intents=intents)

# Dicionários para armazenar o estado por servidor (guild)
conexoes = {}
transcription_tasks = {}

# --- Lógica de Transcrição com Gemini ---

async def transcrever_audio_gemini(audio_bytes):
    """
    Envia os bytes de áudio para a API Gemini e retorna o texto transcrito.
    """
    if not audio_bytes:
        return "[Áudio vazio]"

    uploaded_file = None
    temp_audio_file_path = None
    try:
        # Gemini API funciona melhor com arquivos. Criamos um arquivo temporário para os bytes de áudio.
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_audio_file:
            temp_audio_file.write(audio_bytes)
            temp_audio_file_path = temp_audio_file.name

        # 1. Upload do arquivo para a API do Gemini
        print(f"Fazendo upload do arquivo de áudio temporário: {temp_audio_file_path}")
        uploaded_file = await asyncio.to_thread(
            genai.upload_file,
            path=temp_audio_file_path
        )
        print(f"Upload concluído: {uploaded_file.name}")

        # 2. Criação do modelo e geração do conteúdo
        model = genai.GenerativeModel('models/gemini-1.5-flash-latest')
        print("Gerando conteúdo a partir do áudio...")
        response = await asyncio.to_thread(
            model.generate_content,
            ["Por favor, transcreva o seguinte áudio:", uploaded_file]
        )
        print("Transcrição recebida.")

        return response.text

    except Exception as e:
        print(f"Ocorreu um erro ao transcrever com Gemini: {e}")
        return f"[Erro na transcrição com Gemini: {e}]"
    finally:
        # 3. Limpeza
        if uploaded_file:
            print(f"Deletando arquivo do Gemini: {uploaded_file.name}")
            await asyncio.to_thread(genai.delete_file, name=uploaded_file.name)
        if temp_audio_file_path and os.path.exists(temp_audio_file_path):
            print(f"Deletando arquivo local temporário: {temp_audio_file_path}")
            os.remove(temp_audio_file_path)

# --- Lógica de Gravação Contínua ---

async def periodic_transcription_task(sink: discord.sinks.WaveSink, channel: discord.TextChannel, guild_id: int):
    """
    A cada 4 minutos, processa o áudio gravado e o envia para transcrição.
    """
    while guild_id in conexoes:
        await asyncio.sleep(240)  # Espera 4 minutos (240 segundos)

        if not sink.audio_data:
            print("Nenhum áudio para processar neste intervalo.")
            continue

        await channel.send("`Processando transcrições parciais (últimos 4 minutos)...`")

        # Copia os dados de áudio para processamento e limpa os buffers originais
        audio_data_copy = sink.audio_data.copy()
        for audio in sink.audio_data.values():
            audio.file.seek(0)
            audio.file.truncate(0)

        for user_id, audio in audio_data_copy.items():
            try:
                user = channel.guild.get_member(user_id) or await bot.fetch_user(user_id)
                nome_usuario = user.display_name if user else f"Usuário Desconhecido ({user_id})"
                texto_transcrito = await transcrever_audio_gemini(audio.file.getvalue())

                if texto_transcrito and not texto_transcrito.startswith("["):
                    await channel.send(f"**{nome_usuario} (parcial):**\n>>> {texto_transcrito}")
            except Exception as e:
                print(f"Erro ao processar áudio parcial para o usuário {user_id}: {e}")

# --- Callback Pós-Gravação (Final) ---

async def after_recording_callback(sink: discord.sinks.WaveSink, channel: discord.TextChannel, *args):
    """
    Chamado quando a gravação é interrompida pelo comando /sair.
    Processa o segmento final de áudio.
    """
    await channel.send("`Gravação finalizada. Processando o áudio restante...`")

    for user_id, audio in sink.audio_data.items():
        try:
            user = channel.guild.get_member(user_id) or await bot.fetch_user(user_id)
            nome_usuario = user.display_name if user else f"Usuário Desconhecido ({user_id})"
            texto_transcrito = await transcrever_audio_gemini(audio.file.getvalue())

            if texto_transcrito and not texto_transcrito.startswith("["):
                await channel.send(f"**{nome_usuario} (final):**\n>>> {texto_transcrito}")
        except Exception as e:
            print(f"Erro ao processar áudio final para o usuário {user_id}: {e}")

    try:
        sink.cleanup()
    except discord.sinks.errors.SinkException as e:
        print(f"Aviso: Ocorreu um erro esperado durante a limpeza do sink: {e}")

# --- Comandos do Bot (Slash Commands) ---

@bot.slash_command(name="entrar", description="Conecta o bot, grava e transcreve a cada 4 minutos.")
async def entrar(ctx: discord.ApplicationContext):
    if not ctx.author.voice:
        await ctx.respond("Você não está em um canal de voz.", ephemeral=True)
        return

    voice_channel = ctx.author.voice.channel
    if ctx.voice_client:
        await ctx.respond("Já estou conectado a um canal de voz.", ephemeral=True)
        return

    try:
        vc = await voice_channel.connect()
        conexoes[ctx.guild.id] = vc
    except Exception as e:
        await ctx.respond(f"Ocorreu um erro ao conectar: {e}", ephemeral=True)
        return

    sink = discord.sinks.WaveSink()
    vc.start_recording(
        sink,
        after_recording_callback,
        ctx.channel
    )

    task = bot.loop.create_task(periodic_transcription_task(sink, ctx.channel, ctx.guild.id))
    transcription_tasks[ctx.guild.id] = task

    await ctx.respond(f"Conectado a '{voice_channel.name}'. Gravando e transcrevendo a cada 4 minutos! 🎙️✨")

@bot.slash_command(name="sair", description="Para a gravação, transcreve o áudio final e desconecta.")
async def sair(ctx: discord.ApplicationContext):
    if ctx.guild.id not in conexoes:
        await ctx.respond("Não estou em nenhum canal de voz.", ephemeral=True)
        return

    task = transcription_tasks.pop(ctx.guild.id, None)
    if task:
        task.cancel()

    vc = conexoes[ctx.guild.id]

    # Para a gravação. Isso acionará o after_recording_callback.
    vc.stop_recording()

    # Desconecta do canal de voz
    await vc.disconnect()

    # Remove a conexão da lista de conexões ativas
    del conexoes[ctx.guild.id]

    await ctx.respond("Gravação parada. As transcrições serão enviadas em breve.")

# --- Eventos do Bot ---

@bot.event
async def on_ready():
    """
    Evento que é acionado quando o bot está online e pronto para uso.
    """
    print(f"Bot '{bot.user}' está online e pronto!")
    print("Usando a API do Google Gemini para transcrição.")
    print("-" * 20)


# --- Célula 4: Execução do Bot ---
# Como estamos no Colab, que já tem um loop de eventos asyncio rodando,
# usamos bot.start() com await.

async def main():
    """Função principal para iniciar o bot."""
    print("Iniciando o bot...")
    await bot.start(DISCORD_TOKEN)


# Para rodar no Colab, você pode simplesmente chamar a função main.
# O 'await' no nível superior do notebook cuidará disso.
# Exemplo de como executar na última célula do seu notebook:
#
try:
    await main()
except KeyboardInterrupt:
    print("Bot desligado manualmente.")


# Se você estiver executando este script como um arquivo .py normal,
# you could use the following line at the end:
# bot.run(DISCORD_TOKEN)
# But for Colab, the await bot.start() approach is correct.

AttributeError: module 'discord' has no attribute 'Bot'